# XFLR5 Baseline vs Optimised run comparison

Builds a complete `(Re, alpha)` grid for `alpha in [-30, 30]` with `0.5` step, parses all result CSVs in `Results/Baseline` and `Results/Optimised`, and exports:
- a merged comparison CSV with `NaN` placeholders where convergence is missing
- a convergence summary CSV per Reynolds number

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# ===== User-configurable paths =====
ROOT = Path('/Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil')
BASELINE_DIR = ROOT / 'Results' / 'Baseline'
OPTIMISED_DIR = ROOT / 'Results' / 'Optimised'

OUT_COMPARISON = ROOT / 'Results' / 'xflr5_baseline_vs_optimised_grid.csv'
OUT_CONV_SUMMARY = ROOT / 'Results' / 'xflr5_convergence_summary.csv'

# ===== Expected alpha grid =====
ALPHA_MIN = -30.0
ALPHA_MAX = 30.0
ALPHA_STEP = 0.5
alpha_grid = np.round(np.arange(ALPHA_MIN, ALPHA_MAX + ALPHA_STEP / 2, ALPHA_STEP), 3)
EXPECTED_ALPHA_COUNT = len(alpha_grid)

print('Baseline dir :', BASELINE_DIR)
print('Optimised dir:', OPTIMISED_DIR)
print('Expected alpha points:', EXPECTED_ALPHA_COUNT)

Baseline dir : /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/Baseline
Optimised dir: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/Optimised
Expected alpha points: 121


In [2]:
RE_FROM_NAME = re.compile(r'Re0\.(\d+)')

def parse_re_from_filename(path: Path) -> int:
    m = RE_FROM_NAME.search(path.name)
    if not m:
        raise ValueError(f'Could not parse Reynolds from filename: {path.name}')
    return int(m.group(1))

def parse_xflr5_csv(path: Path, airfoil_label: str) -> pd.DataFrame:
    """
    Parses one XFLR5 CSV with mixed text header + numeric body.
    Keeps only first 10 standard columns matching the header:
      alpha,CL,CD,CDp,Cm,Top Xtr,Bot Xtr,Cpmin,Chinge,XCp
    """
    text = path.read_text(encoding='utf-8', errors='ignore')
    lines = text.splitlines()

    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith('alpha,'):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f'No alpha header found in {path.name}')

    header_cols = [c.strip() for c in lines[header_idx].split(',')]
    keep_cols = header_cols[:10]

    rows = []
    for raw in lines[header_idx + 1:]:
        s = raw.strip()
        if not s:
            continue

        # Skip any accidental non-data lines after header
        if not (s[0].isdigit() or s[0] in '+-.'):
            continue

        parts = [p.strip() for p in raw.split(',')]
        vals = []
        ok = True
        for p in parts:
            if p == '':
                vals.append(np.nan)
            else:
                try:
                    vals.append(float(p))
                except ValueError:
                    ok = False
                    break

        if not ok or len(vals) < 2:
            continue

        # XFLR5 may output extra trailing numeric fields; keep standard first 10
        vals = (vals + [np.nan] * 10)[:10]
        rows.append(vals)

    df = pd.DataFrame(rows, columns=keep_cols)
    if not df.empty:
        df['alpha'] = pd.to_numeric(df['alpha'], errors='coerce').round(3)
        # If duplicate alpha exists, keep first occurrence
        df = df.drop_duplicates(subset=['alpha'], keep='first')

    re_int = parse_re_from_filename(path)
    df['Re'] = re_int
    df['airfoil'] = airfoil_label
    df['source_file'] = path.name
    return df

def parse_folder(folder: Path, airfoil_label: str) -> pd.DataFrame:
    files = sorted(folder.glob('*.csv'))
    all_parts = []
    for f in files:
        try:
            all_parts.append(parse_xflr5_csv(f, airfoil_label))
        except Exception as exc:
            print(f'[WARN] Skipping {f.name}: {exc}')

    if not all_parts:
        return pd.DataFrame(columns=['alpha','CL','CD','CDp','Cm','Top Xtr','Bot Xtr','Cpmin','Chinge','XCp','Re','airfoil','source_file'])

    return pd.concat(all_parts, ignore_index=True)

In [6]:
baseline_raw = parse_folder(BASELINE_DIR, 'Baseline')
optimised_raw = parse_folder(OPTIMISED_DIR, 'Optimised')

print('Baseline rows parsed :', len(baseline_raw))
print('Optimised rows parsed:', len(optimised_raw))

# Build Reynolds union from filenames (captures fully crashed/empty CSV runs too)
def re_values_from_filenames(folder: Path) -> set[int]:
    vals = set()
    for f in sorted(folder.glob('*.csv')):
        try:
            vals.add(parse_re_from_filename(f))
        except Exception:
            pass
    return vals

re_baseline = re_values_from_filenames(BASELINE_DIR)
re_optimised = re_values_from_filenames(OPTIMISED_DIR)
all_re = sorted(re_baseline | re_optimised)

print('Reynolds count in union:', len(all_re))
print('Reynolds values:', all_re)

Baseline rows parsed : 3327
Optimised rows parsed: 3237
Reynolds count in union: 32
Reynolds values: [160, 641, 1597, 3020, 4896, 7206, 9930, 13039, 16506, 20295, 24371, 28694, 33222, 37913, 42721, 47599, 52501, 57379, 62187, 66878, 71406, 75729, 79805, 83594, 87061, 90170, 92894, 95204, 97080, 98503, 99459, 99940]


/var/folders/qg/4c4w3kmn1xqgh6mp5z7g0s580000gp/T/ipykernel_1700/3108366011.py:84: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_parts, ignore_index=True)
/var/folders/qg/4c4w3kmn1xqgh6mp5z7g0s580000gp/T/ipykernel_1700/3108366011.py:84: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_parts, ignore_index=True)


In [7]:
def to_full_grid(df_raw: pd.DataFrame, prefix: str, all_re: list[int], alpha_grid: np.ndarray) -> pd.DataFrame:
    idx = pd.MultiIndex.from_product([all_re, alpha_grid], names=['Re', 'alpha'])

    keep = ['CL','CD','CDp','Cm','Top Xtr','Bot Xtr','Cpmin','Chinge','XCp']
    if df_raw.empty:
        out = pd.DataFrame(index=idx, columns=[f'{prefix}_{c}' for c in keep], dtype=float)
        out[f'{prefix}_has_data'] = False
        return out

    work = df_raw.copy()
    work['Re'] = pd.to_numeric(work['Re'], errors='coerce').astype('Int64')
    work['alpha'] = pd.to_numeric(work['alpha'], errors='coerce').round(3)

    # keep one row per (Re, alpha)
    work = work.dropna(subset=['Re','alpha']).drop_duplicates(subset=['Re','alpha'], keep='first')
    work = work.set_index(['Re','alpha'])[keep]

    out = work.reindex(idx)
    out = out.rename(columns={c: f'{prefix}_{c}' for c in keep})

    # per-point convergence proxy: CL present => row converged/exported
    out[f'{prefix}_has_data'] = out[f'{prefix}_CL'].notna()
    return out

baseline_grid = to_full_grid(baseline_raw, 'baseline', all_re, alpha_grid)
optimised_grid = to_full_grid(optimised_raw, 'optimised', all_re, alpha_grid)

comparison = baseline_grid.join(optimised_grid, how='outer').reset_index()

# Helpful per-row status labels
comparison['status'] = np.select(
    [
        comparison['baseline_has_data'] & comparison['optimised_has_data'],
        comparison['baseline_has_data'] & ~comparison['optimised_has_data'],
        ~comparison['baseline_has_data'] & comparison['optimised_has_data'],
    ],
    [
        'both_converged',
        'baseline_only',
        'optimised_only',
    ],
    default='none_converged'
)

comparison.to_csv(OUT_COMPARISON, index=False)
print('Saved comparison CSV:', OUT_COMPARISON)
print('Rows:', len(comparison))
comparison.head(8)

Saved comparison CSV: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/xflr5_baseline_vs_optimised_grid.csv
Rows: 3872


,Re,alpha,baseline_CL,baseline_CD,baseline_CDp,baseline_Cm,baseline_Top Xtr,baseline_Bot Xtr,baseline_Cpmin,baseline_Chinge,...,optimised_CD,optimised_CDp,optimised_Cm,optimised_Top Xtr,optimised_Bot Xtr,optimised_Cpmin,optimised_Chinge,optimised_XCp,optimised_has_data,status
0,160,-30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,none_converged
1,160,-29.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,none_converged
2,160,-29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,none_converged
3,160,-28.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,none_converged
4,160,-28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,none_converged
5,160,-27.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,none_converged
6,160,-27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,none_converged
7,160,-26.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,none_converged


In [8]:
# Convergence summary (% over expected 121 alpha points)
summary = (
    comparison.groupby('Re', as_index=False)
    .agg(
        baseline_converged=('baseline_has_data', 'sum'),
        optimised_converged=('optimised_has_data', 'sum')
    )
)
summary['expected_points'] = EXPECTED_ALPHA_COUNT
summary['baseline_convergence_pct'] = 100.0 * summary['baseline_converged'] / summary['expected_points']
summary['optimised_convergence_pct'] = 100.0 * summary['optimised_converged'] / summary['expected_points']

summary.to_csv(OUT_CONV_SUMMARY, index=False)
print('Saved convergence summary CSV:', OUT_CONV_SUMMARY)
summary

Saved convergence summary CSV: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/xflr5_convergence_summary.csv


,Re,baseline_converged,optimised_converged,expected_points,baseline_convergence_pct,optimised_convergence_pct
0,160,0,0,121,0.000000,0.000000
1,641,121,120,121,100.000000,99.173554
2,1597,121,120,121,100.000000,99.173554
3,3020,120,121,121,99.173554,100.000000
4,4896,121,120,121,100.000000,99.173554
5,7206,120,119,121,99.173554,98.347107
6,9930,117,118,121,96.694215,97.520661
7,13039,120,120,121,99.173554,99.173554
8,16506,120,116,121,99.173554,95.867769
9,20295,112,111,121,92.561983,91.735537


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Load comparison CSV (created by previous cells)
csv_path = Path('/Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/xflr5_baseline_vs_optimised_grid.csv')
if not csv_path.exists():
    raise FileNotFoundError(f'Comparison CSV not found: {csv_path}')

df_cmp = pd.read_csv(csv_path)
re_values = sorted(df_cmp['Re'].dropna().astype(int).unique().tolist())

if not re_values:
    raise ValueError('No Reynolds values found in comparison CSV.')

# Consistent colors across all subplots
BASELINE_COLOR = 'tab:blue'
OPTIMISED_COLOR = 'tab:orange'

def _add_missing_markers(ax, x_missing, y_ref, color, marker, label):
    if len(x_missing) == 0:
        return
    y0, y1 = ax.get_ylim()
    if np.isfinite(y0) and np.isfinite(y1):
        y_mark = y0 + 0.05 * (y1 - y0)
    else:
        finite_ref = y_ref[np.isfinite(y_ref)]
        y_mark = float(finite_ref.min()) if len(finite_ref) > 0 else 0.0
    ax.scatter(
        x_missing,
        np.full_like(x_missing, y_mark, dtype=float),
        marker=marker,
        s=28,
        color=color,
        alpha=0.9,
        label=label,
        zorder=4,
    )

def _safe_ratio(cl, cd):
    out = np.full_like(cl, np.nan, dtype=float)
    valid = np.isfinite(cl) & np.isfinite(cd) & (np.abs(cd) > 1e-12)
    out[valid] = cl[valid] / cd[valid]
    return out

def plot_for_re(selected_re):
    sub = df_cmp[df_cmp['Re'] == int(selected_re)].copy().sort_values('alpha')

    alpha = sub['alpha'].to_numpy(dtype=float)

    b_cl = sub['baseline_CL'].to_numpy(dtype=float)
    o_cl = sub['optimised_CL'].to_numpy(dtype=float)
    b_cd = sub['baseline_CD'].to_numpy(dtype=float)
    o_cd = sub['optimised_CD'].to_numpy(dtype=float)

    b_clcd = _safe_ratio(b_cl, b_cd)
    o_clcd = _safe_ratio(o_cl, o_cd)

    # Missing locations (NaN) -> line cuts + explicit markers
    miss_b_cl = alpha[np.isnan(b_cl)]
    miss_o_cl = alpha[np.isnan(o_cl)]
    miss_b_cd = alpha[np.isnan(b_cd)]
    miss_o_cd = alpha[np.isnan(o_cd)]
    miss_b_clcd = alpha[np.isnan(b_clcd)]
    miss_o_clcd = alpha[np.isnan(o_clcd)]

    fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)

    # --- CL plot ---
    axes[0].plot(alpha, b_cl, label='Baseline CL', color=BASELINE_COLOR, linewidth=2)
    axes[0].plot(alpha, o_cl, label='Optimised CL', color=OPTIMISED_COLOR, linewidth=2)
    axes[0].set_ylabel('CL')
    axes[0].set_title(f'Baseline vs Optimised at Re={int(selected_re)}')
    axes[0].grid(True, alpha=0.3)
    _add_missing_markers(axes[0], miss_b_cl, b_cl, BASELINE_COLOR, 'x', 'Baseline CL missing')
    _add_missing_markers(axes[0], miss_o_cl, o_cl, OPTIMISED_COLOR, '+', 'Optimised CL missing')
    axes[0].legend(loc='best')

    # --- CD plot ---
    axes[1].plot(alpha, b_cd, label='Baseline CD', color=BASELINE_COLOR, linewidth=2)
    axes[1].plot(alpha, o_cd, label='Optimised CD', color=OPTIMISED_COLOR, linewidth=2)
    axes[1].set_ylabel('CD')
    axes[1].grid(True, alpha=0.3)
    _add_missing_markers(axes[1], miss_b_cd, b_cd, BASELINE_COLOR, 'x', 'Baseline CD missing')
    _add_missing_markers(axes[1], miss_o_cd, o_cd, OPTIMISED_COLOR, '+', 'Optimised CD missing')
    axes[1].legend(loc='best')

    # --- CL/CD plot ---
    axes[2].plot(alpha, b_clcd, label='Baseline CL/CD', color=BASELINE_COLOR, linewidth=2)
    axes[2].plot(alpha, o_clcd, label='Optimised CL/CD', color=OPTIMISED_COLOR, linewidth=2)
    axes[2].set_xlabel('alpha (deg)')
    axes[2].set_ylabel('CL/CD')
    axes[2].grid(True, alpha=0.3)
    _add_missing_markers(axes[2], miss_b_clcd, b_clcd, BASELINE_COLOR, 'x', 'Baseline CL/CD missing')
    _add_missing_markers(axes[2], miss_o_clcd, o_clcd, OPTIMISED_COLOR, '+', 'Optimised CL/CD missing')
    axes[2].legend(loc='best')

    plt.tight_layout()
    plt.show()

    print(f'Baseline converged points: {np.sum(~np.isnan(b_cl))}/{len(alpha)}')
    print(f'Optimised converged points: {np.sum(~np.isnan(o_cl))}/{len(alpha)}')

re_dropdown = widgets.Dropdown(
    options=re_values,
    value=re_values[0],
    description='Re:',
    layout=widgets.Layout(width='280px')
)

ui = widgets.interactive_output(plot_for_re, {'selected_re': re_dropdown})
display(re_dropdown, ui)

Dropdown(description='Re:', layout=Layout(width='280px'), options=(160, 641, 1597, 3020, 4896, 7206, 9930, 130…

Output()

In [12]:
# Build the same comparison CSV from finer-delta runs (sampled every 0.5 alpha)
FINER_DIR = ROOT / 'Results' / 'finer_delta'
OUT_COMPARISON_FINER = ROOT / 'Results' / 'xflr5_baseline_vs_optimised_grid_finer.csv'

def parse_folder_with_prefix(folder: Path, airfoil_label: str, prefix: str) -> pd.DataFrame:
    files = sorted(folder.glob(f'{prefix}*.csv'))
    all_parts = []
    for f in files:
        try:
            all_parts.append(parse_xflr5_csv(f, airfoil_label))
        except Exception as exc:
            print(f'[WARN] Skipping {f.name}: {exc}')

    if not all_parts:
        return pd.DataFrame(columns=['alpha','CL','CD','CDp','Cm','Top Xtr','Bot Xtr','Cpmin','Chinge','XCp','Re','airfoil','source_file'])

    return pd.concat(all_parts, ignore_index=True)

def keep_half_degree_alpha(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw.empty:
        return df_raw.copy()
    out = df_raw.copy()
    out['alpha'] = pd.to_numeric(out['alpha'], errors='coerce')
    nearest_half = np.round(out['alpha'] * 2.0) / 2.0
    keep_mask = np.isclose(out['alpha'], nearest_half, atol=1e-4, equal_nan=False)
    out = out[keep_mask].copy()
    out['alpha'] = nearest_half[keep_mask].round(3)
    return out

def re_values_from_filenames_with_prefix(folder: Path, prefix: str) -> set[int]:
    vals = set()
    for f in sorted(folder.glob(f'{prefix}*.csv')):
        try:
            vals.add(parse_re_from_filename(f))
        except Exception:
            pass
    return vals

if not FINER_DIR.exists():
    raise FileNotFoundError(f'Finer directory not found: {FINER_DIR}')

baseline_finer_raw = parse_folder_with_prefix(FINER_DIR, 'Baseline', 'Baseline_')
optimised_finer_raw = parse_folder_with_prefix(FINER_DIR, 'Optimised', 'Optimised_')

baseline_finer_raw = keep_half_degree_alpha(baseline_finer_raw)
optimised_finer_raw = keep_half_degree_alpha(optimised_finer_raw)

re_baseline_finer = re_values_from_filenames_with_prefix(FINER_DIR, 'Baseline_')
re_optimised_finer = re_values_from_filenames_with_prefix(FINER_DIR, 'Optimised_')
all_re_finer = sorted(re_baseline_finer | re_optimised_finer)

baseline_finer_grid = to_full_grid(baseline_finer_raw, 'baseline', all_re_finer, alpha_grid)
optimised_finer_grid = to_full_grid(optimised_finer_raw, 'optimised', all_re_finer, alpha_grid)

comparison_finer = baseline_finer_grid.join(optimised_finer_grid, how='outer').reset_index()
comparison_finer['status'] = np.select(
    [
        comparison_finer['baseline_has_data'] & comparison_finer['optimised_has_data'],
        comparison_finer['baseline_has_data'] & ~comparison_finer['optimised_has_data'],
        ~comparison_finer['baseline_has_data'] & comparison_finer['optimised_has_data'],
    ],
    [
        'both_converged',
        'baseline_only',
        'optimised_only',
    ],
    default='none_converged'
 )

comparison_finer.to_csv(OUT_COMPARISON_FINER, index=False)
print('Saved finer comparison CSV:', OUT_COMPARISON_FINER)
print('Rows:', len(comparison_finer))
comparison_finer.head(8)

Saved finer comparison CSV: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/xflr5_baseline_vs_optimised_grid_finer.csv
Rows: 3872


,Re,alpha,baseline_CL,baseline_CD,baseline_CDp,baseline_Cm,baseline_Top Xtr,baseline_Bot Xtr,baseline_Cpmin,baseline_Chinge,...,optimised_CD,optimised_CDp,optimised_Cm,optimised_Top Xtr,optimised_Bot Xtr,optimised_Cpmin,optimised_Chinge,optimised_XCp,optimised_has_data,status
0,160,-30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,none_converged
1,160,-29.5,-0.5793,0.51035,0.38615,0.0357,1.0,1.0,-1.3668,0.0,...,0.52462,0.37426,0.1033,1.0,1.0,-2.3242,0.0,0.0,True,both_converged
2,160,-29.0,-0.5746,0.50560,0.38021,0.0330,1.0,1.0,-1.3752,0.0,...,0.51944,0.36826,0.0986,1.0,1.0,-2.3226,0.0,0.0,True,both_converged
3,160,-28.5,-0.5697,0.50088,0.37431,0.0303,1.0,1.0,-1.3862,0.0,...,0.51428,0.36232,0.0939,1.0,1.0,-2.3201,0.0,0.0,True,both_converged
4,160,-28.0,-0.5648,0.49620,0.36845,0.0277,1.0,1.0,-1.3962,0.0,...,0.50916,0.35644,0.0893,1.0,1.0,-2.3204,0.0,0.0,True,both_converged
5,160,-27.5,-0.5596,0.49141,0.36233,0.0251,1.0,1.0,-1.4047,0.0,...,0.50397,0.35031,0.0849,1.0,1.0,-2.3219,0.0,0.0,True,both_converged
6,160,-27.0,-0.5543,0.48668,0.35642,0.0225,1.0,1.0,-1.4137,0.0,...,0.49883,0.34445,0.0805,1.0,1.0,-2.3246,0.0,0.0,True,both_converged
7,160,-26.5,-0.5490,0.48200,0.35057,0.0200,1.0,1.0,-1.4216,0.0,...,0.49371,0.33864,0.0761,1.0,1.0,-2.3266,0.0,0.0,True,both_converged
